In [ ]:
# =========================================================
# IMPORTS
# =========================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier

# =========================================================
# DEFINE FINAL MODELS
# =========================================================

models = {

    # =====================================================
    # RANDOM FOREST
    # =====================================================

    "Random Forest": RandomForestClassifier(

        n_estimators=300,

        max_depth=None,

        min_samples_split=10,

        min_samples_leaf=4,

        max_features="sqrt",

        class_weight="balanced",

        random_state=42,

        n_jobs=-1
    ),

    # =====================================================
    # SVM
    # StandardScaler + RBF Kernel
    # =====================================================

    "SVM": Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            SVC(

                kernel="rbf",

                C=10,

                gamma="scale",

                class_weight="balanced",

                probability=True,

                random_state=42
            )
        )

    ]),

    # =====================================================
    # kNN
    # StandardScaler + Euclidean Distance
    # =====================================================

    "kNN": Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            KNeighborsClassifier(

                n_neighbors=3,

                weights="uniform",

                metric="euclidean"
            )
        )

    ]),

    # =====================================================
    # XGBOOST
    # =====================================================

    "XGBoost": XGBClassifier(

        n_estimators=300,

        max_depth=10,

        learning_rate=0.1,

        min_child_weight=3,

        subsample=0.8,

        colsample_bytree=1.0,

        objective="multi:softprob",

        num_class=3,

        eval_metric="mlogloss",

        random_state=42,

        tree_method="hist"
    )
}

# =========================================================
# CHECK MODELS
# =========================================================

print("FINAL MODELS")
print("============")

for name in models.keys():

    print(f"✓ {name}")

In [ ]:
from sklearn.base import clone

# =========================================================
# TRAIN MODELS WITHOUT SMOTE
# =========================================================

fitted_models_no_smote = {}

for name, model in models.items():

    print(f"Training {name} (No SMOTE)...")

    model_no = clone(model)

    model_no.fit(
        X_train,
        y_train
    )

    fitted_models_no_smote[name] = model_no

# =========================================================
# TRAIN MODELS WITH SMOTE
# =========================================================

fitted_models_smote = {}

for name, model in models.items():

    print(f"Training {name} (SMOTE)...")

    model_sm = clone(model)

    model_sm.fit(
        X_train_smote,
        y_train_smote
    )

    fitted_models_smote[name] = model_sm

In [ ]:
print(
    fitted_models_no_smote["Random Forest"]
    is
    fitted_models_smote["Random Forest"]
)

print(
    fitted_models_no_smote["XGBoost"]
    is
    fitted_models_smote["XGBoost"]
)

print(
    fitted_models_no_smote["SVM"]
    is
    fitted_models_smote["SVM"]
)

print(
    fitted_models_no_smote["kNN"]
    is
    fitted_models_smote["kNN"]
)

In [ ]:
# =========================================================
# TRAINED MODELS SUMMARY
# =========================================================

print("\n=================================================")
print("MODELS TRAINED WITHOUT SMOTE")
print("=================================================")

if len(fitted_models_no_smote) == 0:

    print("No models found.")

else:

    print(f"Total Models: {len(fitted_models_no_smote)}\n")

    for i, model_name in enumerate(
        fitted_models_no_smote.keys(),
        start=1
    ):

        print(f"{i}. {model_name}")

# =========================================================

print("\n=================================================")
print("MODELS TRAINED WITH SMOTE")
print("=================================================")

if len(fitted_models_smote) == 0:

    print("No models found.")

else:

    print(f"Total Models: {len(fitted_models_smote)}\n")

    for i, model_name in enumerate(
        fitted_models_smote.keys(),
        start=1
    ):

        print(f"{i}. {model_name}")

In [ ]:
# =========================================================
# COMPLETE MODEL EVALUATION
# Metrics based on methodology table
# =========================================================

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# =========================================================
# STORAGE
# =========================================================

evaluation_results = []

predictions_no_smote = {}
predictions_smote = {}

confusion_matrices_no_smote = {}
confusion_matrices_smote = {}

reports_no_smote = {}
reports_smote = {}

# =========================================================
# EVALUATION FUNCTION
# =========================================================

def evaluate_model_group(
    model_dict,
    smote_status
):

    for name, model in model_dict.items():

        print("\n=================================================")
        print(f"MODEL : {name}")
        print(f"SMOTE : {smote_status}")
        print("=================================================")

        # =================================================
        # PREDICTIONS
        # =================================================

        y_pred = model.predict(X_test)

        # =================================================
        # SAVE PREDICTIONS
        # =================================================

        if smote_status == "No":

            predictions_no_smote[name] = y_pred

        else:

            predictions_smote[name] = y_pred

        # =================================================
        # OVERALL METRICS
        # =================================================

        accuracy = accuracy_score(
            y_test,
            y_pred
        )

        macro_precision = precision_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )

        macro_recall = recall_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )

        macro_f1 = f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )

        weighted_f1 = f1_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )

        # =================================================
        # PER-CLASS METRICS
        # =================================================

        precision_class = precision_score(
            y_test,
            y_pred,
            average=None,
            zero_division=0
        )

        recall_class = recall_score(
            y_test,
            y_pred,
            average=None,
            zero_division=0
        )

        f1_class = f1_score(
            y_test,
            y_pred,
            average=None,
            zero_division=0
        )

        # =================================================
        # REPORT
        # =================================================

        report = classification_report(
            y_test,
            y_pred,
            output_dict=True,
            zero_division=0
        )

        if smote_status == "No":

            reports_no_smote[name] = report

        else:

            reports_smote[name] = report

        # =================================================
        # CONFUSION MATRIX
        # =================================================

        cm = confusion_matrix(
            y_test,
            y_pred
        )

        if smote_status == "No":

            confusion_matrices_no_smote[name] = cm

        else:

            confusion_matrices_smote[name] = cm

        # =================================================
        # PRINT RESULTS
        # =================================================

        print(f"Accuracy         : {accuracy:.4f}")
        print(f"Macro Precision  : {macro_precision:.4f}")
        print(f"Macro Recall     : {macro_recall:.4f}")
        print(f"Macro F1-score   : {macro_f1:.4f}")
        print(f"Weighted F1-score: {weighted_f1:.4f}")

        print("\nConfusion Matrix")
        print(cm)

        # =================================================
        # SAVE RESULTS
        # =================================================

        evaluation_results.append({

            "Model": name,
            "SMOTE": smote_status,

            "Accuracy": accuracy,

            "Macro_Precision": macro_precision,
            "Macro_Recall": macro_recall,
            "Macro_F1": macro_f1,
            "Weighted_F1": weighted_f1,

            "Precision_Class0": precision_class[0],
            "Precision_Class1": precision_class[1],
            "Precision_Class2": precision_class[2],

            "Recall_Class0": recall_class[0],
            "Recall_Class1": recall_class[1],
            "Recall_Class2": recall_class[2],

            "F1_Class0": f1_class[0],
            "F1_Class1": f1_class[1],
            "F1_Class2": f1_class[2]
        })

# =========================================================
# EVALUATE NO-SMOTE
# =========================================================

print("\n#################################################")
print("EVALUATION WITHOUT SMOTE")
print("#################################################")

evaluate_model_group(
    fitted_models_no_smote,
    "No"
)

# =========================================================
# EVALUATE SMOTE
# =========================================================

print("\n#################################################")
print("EVALUATION WITH SMOTE")
print("#################################################")

evaluate_model_group(
    fitted_models_smote,
    "Yes"
)

# =========================================================
# CREATE RESULTS TABLE
# =========================================================

results_df = pd.DataFrame(
    evaluation_results
)

results_df = results_df.sort_values(
    by="Macro_F1",
    ascending=False
).reset_index(drop=True)

results_df = results_df.round(4)

# =========================================================
# DISPLAY
# =========================================================

print("\n=================================================")
print("FINAL MODEL COMPARISON")
print("=================================================")

display(results_df)

# =========================================================
# SAVE
# =========================================================

results_df.to_csv(
    "Model_Comparison_SMOTE_vs_NoSMOTE.csv",
    index=False
)

print("\nSaved:")
print("Model_Comparison_SMOTE_vs_NoSMOTE.csv")

# =========================================================
# BEST MODEL
# =========================================================

best_model_row = results_df.iloc[0]

print("\n=================================================")
print("BEST MODEL")
print("=================================================")

print(f"Model       : {best_model_row['Model']}")
print(f"SMOTE       : {best_model_row['SMOTE']}")
print(f"Accuracy    : {best_model_row['Accuracy']:.4f}")
print(f"Macro F1    : {best_model_row['Macro_F1']:.4f}")

# =========================================================
# SAVE BEST MODEL OBJECT
# =========================================================

best_model_name = best_model_row["Model"]
best_model_smote = best_model_row["SMOTE"]

if best_model_smote == "Yes":

    best_model = fitted_models_smote[
        best_model_name
    ]

else:

    best_model = fitted_models_no_smote[
        best_model_name
    ]

print("\nBest model saved as:")
print("best_model")

In [ ]:
# =========================================================
# IMPORT LIBRARY
# =========================================================

import joblib
import os

# =========================================================
# MAIN DIRECTORY
# =========================================================

main_dir = "saved_models"

os.makedirs(main_dir, exist_ok=True)

# =========================================================
# SAFE FUNCTION TO SAVE MODELS
# =========================================================

def save_models(model_dict, save_dir, suffix):

    os.makedirs(save_dir, exist_ok=True)

    if model_dict is None or len(model_dict) == 0:
        print(f"No models found in {save_dir}")
        return

    for name, model in model_dict.items():

        safe_name = name.replace(" ", "_").replace("/", "_")

        filename = f"{safe_name}_{suffix}.pkl"

        filepath = os.path.join(save_dir, filename)

        joblib.dump(model, filepath)

        print(f"Saved: {filepath}")


# =========================================================
# SAVE NO-SMOTE MODELS
# =========================================================

save_models(
    fitted_models_no_smote,
    os.path.join(main_dir, "No_SMOTE"),
    "NoSMOTE"
)

# =========================================================
# SAVE SMOTE MODELS
# =========================================================

save_models(
    fitted_models_smote,
    os.path.join(main_dir, "SMOTE"),
    "SMOTE"
)

# =========================================================
# DONE
# =========================================================

print("\n====================================")
print("ALL MODELS SAVED SUCCESSFULLY")
print("====================================")